# Complete Text Feature Engineering Assignment

In [37]:

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [38]:

reviews = ["This product is amazing and works perfectly",
"Worst product ever waste of money",
"Good quality and fast delivery",
"Not satisfied with the performance",
"Excellent build quality and value for money",
"Bad experience not recommended",
"Very happy with this purchase",
"Product is okay not great",
"Superb quality highly recommended",
"Terrible product broke in one day"] * 15

df = pd.DataFrame({"review_text": reviews})
df.head()


,review_text
0,This product is amazing and works perfectly
1,Worst product ever waste of money
2,Good quality and fast delivery
3,Not satisfied with the performance
4,Excellent build quality and value for money


In [39]:

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

df['clean_text'] = df['review_text'].apply(clean_text)


In [40]:

df['tokens'] = df['clean_text'].apply(lambda x: x.split())


In [41]:

all_words = [word for tokens in df['tokens'] for word in tokens]
vocab = list(set(all_words))

print("Vocabulary Size:", len(vocab))


Vocabulary Size: 40


In [42]:

def one_hot(text):
    words = text.split()
    return [1 if word in words else 0 for word in vocab]

df['ohe'] = df['clean_text'].apply(one_hot)
print("OHE sample:", df['ohe'][0])


OHE sample: [1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0]


In [43]:

cv = CountVectorizer()
bow = cv.fit_transform(df['clean_text'])
print("BoW Shape:", bow.shape)


BoW Shape: (150, 40)


In [44]:

tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df['clean_text'])
print("TF-IDF Shape:", tfidf_matrix.shape)


TF-IDF Shape: (150, 40)


In [45]:

def sparsity(matrix):
    return 1 - (matrix.count_nonzero()/(matrix.shape[0]*matrix.shape[1]))

print("BoW Sparsity:", sparsity(bow))
print("TF-IDF Sparsity:", sparsity(tfidf_matrix))


BoW Sparsity: 0.865
TF-IDF Sparsity: 0.865


In [46]:

df['label'] = df['review_text'].apply(lambda x: 1 if "good" in x.lower() or "excellent" in x.lower() or "happy" in x.lower() else 0)


In [47]:

X_train, X_test, y_train, y_test = train_test_split(bow, df['label'], test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy BoW:", accuracy_score(y_test, pred))


Accuracy BoW: 1.0


In [48]:

X_train, X_test, y_train, y_test = train_test_split(tfidf_matrix, df['label'], test_size=0.2)

model = LogisticRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)
print("Accuracy TF-IDF:", accuracy_score(y_test, pred))


Accuracy TF-IDF: 1.0



## Comparison

- OHE: simple but high dimensional
- BoW: frequency-based but no semantics
- TF-IDF: highlights important words

## Answers

- BoW fails: cannot capture meaning (good vs excellent)
- Use BoW: simple models
- Use TF-IDF: search/ranking
- TF-IDF limitation: no context understanding
